# EDA Ringkas - NYC Yellow Taxi Januari–Juni 2026
**Analisis Spasial-Temporal dan Prediksi Taxi Demand di NYC**

> **Revisi tanpa mengubah kerangka 5 EDA sebelumnya.** Dataset mengikuti folder lokal `../datasets/raw/2026/` yang kamu gunakan. Notebook ini merupakan EDA berbasis **raw + filter analitis sementara**; **jalankan/validasi ulang menggunakan `main_trips` final** setelah Anggota 1 menyerahkannya. Output lama dibersihkan agar tidak terbaca sebagai hasil final.

| # | EDA awal yang dipertahankan | Tambahan agar sesuai TODO |
|---|---|---|
| 1 | Pola waktu, heatmap, Mann–Whitney | Jumlah/grafik bulanan, pengecekan kalender |
| 2 | Top-K, Pareto, Gini | Distribusi per borough dan target pickup |
| 3 | Asal-tujuan borough | Peta pickup centroid (HTML), jalur OD centroid |
| 4 | `cbd_congestion_fee` deskriptif | Denominator fee teramati, bukan asumsi nol |
| 5 | Indeks potensi pendapatan | Distribusi jarak/durasi/tarif, sebelum-sesudah cleaning |

**RQ1:** pola pickup menurut jam/hari/bulan. **RQ4:** zona dan waktu prioritas berdasarkan volume pickup. Jumlah pickup merepresentasikan *served demand*, bukan seluruh permintaan yang tidak terlayani.

## 0. Persiapan

Memuat library, menyiapkan path, serta fungsi penyimpanan grafik. **Path dataset sengaja mengikuti notebook lokalmu**, sehingga tidak perlu memindahkan enam file Parquet.

In [1]:
import sys, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------ KONFIGURASI (SAMA DENGAN LOKASI LOKALMU) ------------------
DATA_DIR     = Path("../datasets")
FILE_PATTERN = "raw/{tahun}/yellow_tripdata_{tahun}-{bulan:02d}.parquet"
LOOKUP_CSV   = DATA_DIR / "zone" / "taxi_zone_lookup.csv"
OUT_DIR      = Path("../out/output_eda")
FIG_DIR      = OUT_DIR / "figures"
HTML_DIR     = Path("../out/html")
TAHUN        = 2026
BULAN        = [1, 2, 3, 4, 5, 6]
CEK_DUPLIKAT = True
SEED         = 42
ALPHA        = 0.05
# ----------------------------------------------------------------------------

# Fallback jika file kalender Anggota 4 belum ada. Ganti setelah diserahkan.
HARI_LIBUR = {
    "2026-01-01": "New Year's Day", "2026-01-19": "MLK Day",
    "2026-02-16": "Presidents' Day", "2026-05-25": "Memorial Day",
    "2026-06-19": "Juneteenth"
}
np.random.seed(SEED)
for p in (OUT_DIR, FIG_DIR, HTML_DIR): p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12})
NAMA_HARI  = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
NAMA_BULAN = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"Mei", 6:"Jun"}

def simpan(nama):
    plt.savefig(FIG_DIR / f"{nama}.png", dpi=150, bbox_inches="tight")

def ribuan(x, pos=None):
    return f"{x:,.0f}"

# File kalender opsional: date/tanggal dan is_holiday/libur, atau semua baris dianggap libur.
KALENDER_CSV = DATA_DIR / "external" / "holidays_2026.csv"
if KALENDER_CSV.exists():
    cal = pd.read_csv(KALENDER_CSV)
    cols = {c.lower().strip():c for c in cal.columns}
    cdate = cols.get("date", cols.get("tanggal"))
    cflag = cols.get("is_holiday", cols.get("libur"))
    if cdate:
        if cflag:
            f = cal[cflag].astype(str).str.lower().isin(["true", "1", "yes", "ya"])
            cal = cal.loc[f]
        tanggal_libur = set(pd.to_datetime(cal[cdate], errors="coerce").dt.normalize().dropna())
        print("Kalender eksternal dipakai:", KALENDER_CSV)
    else:
        tanggal_libur = set(pd.to_datetime(list(HARI_LIBUR)))
        print("[!] Kalender eksternal tidak memiliki kolom tanggal; memakai fallback.")
else:
    tanggal_libur = set(pd.to_datetime(list(HARI_LIBUR)))
    print("[!] Kalender anggota 4 belum ada; memakai daftar federal holidays sementara.")
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| seed:", SEED)
print("Direktori dataset:", DATA_DIR.resolve())

[!] Kalender anggota 4 belum ada; memakai daftar federal holidays sementara.
Python: 3.13.12 | pandas: 3.0.1 | seed: 42
Direktori dataset: C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets


## Prasyarat: Pemuatan dan Pembersihan Ringan

Alur pemuatan asli tetap dipertahankan: **baca data satu bulan per iterasi**, ringkas menjadi data **zona-jam (`zh_all`)** dan **asal-tujuan (`od_all`)** supaya hemat RAM.

Perbaikan penting: **validitas demand** hanya memerlukan waktu dan zona pickup yang sah. Masalah durasi/jarak/tarif tidak otomatis membuang pickup valid. **Validitas revenue** diperiksa terpisah. Filter masih bersifat *prototipe* dan harus disamakan dengan `docs/analysis_plan.md` serta `main_trips` final sebelum kesimpulan dikunci.

In [ ]:
if not LOOKUP_CSV.exists():
    raise FileNotFoundError(f"Lookup zona tidak ditemukan: {LOOKUP_CSV.resolve()}")
lookup = pd.read_csv(LOOKUP_CSV)
lookup.columns = [str(c).strip() for c in lookup.columns]
for col in ("LocationID", "Borough", "Zone"):
    if col not in lookup.columns:
        raise KeyError(f"Kolom '{col}' tidak ditemukan dalam taxi_zone_lookup.csv")
lookup["LocationID"] = pd.to_numeric(lookup["LocationID"], errors="coerce")
lookup = lookup.dropna(subset=["LocationID"]).drop_duplicates("LocationID").copy()
lookup["LocationID"] = lookup["LocationID"].astype(int)
lookup["Borough"] = lookup["Borough"].fillna("Tidak diketahui")
lookup["Zone"] = lookup["Zone"].fillna("?")
lookup["label"] = lookup["Zone"] + " (" + lookup["LocationID"].astype(str) + ")"
# Sesuai filtering sementara lama, area Unknown 264/265 tidak dipakai.
# Keputusan final tetap mengikuti Anggota 1 / analysis_plan (masih Draft).
ZONA_VALID = set(lookup.loc[~lookup["LocationID"].isin([264,265]), "LocationID"])
b_map = lookup.set_index("LocationID")["Borough"]
lbl_map = lookup.set_index("LocationID")["label"]
def nama_zona(z): return lbl_map.get(z, f"Zona {z}")

KOLOM_WAJIB = ["tpep_pickup_datetime", "PULocationID", "DOLocationID",
               "tpep_dropoff_datetime", "trip_distance", "fare_amount", "total_amount"]
ATURAN_BUANG = ["Pickup di luar bulan berkas / null", "Zona pickup invalid/Unknown"]
# Sampel reproducible untuk histogram (data penuh tetap dipakai untuk seluruh agregasi).
MAX_SAMPEL_BULAN = 20000

def proses_bulan(bulan):
    path = DATA_DIR / FILE_PATTERN.format(tahun=TAHUN, bulan=bulan)
    if not path.exists():
        raise FileNotFoundError(f"Dataset bulan {bulan:02d} tidak ditemukan: {path.resolve()}")
    df = pd.read_parquet(path)
    df.columns = ["Airport_fee" if c.lower()=="airport_fee" else c for c in df.columns]
    kurang = [c for c in KOLOM_WAJIB if c not in df.columns]
    if kurang: raise KeyError(f"Kolom pada {path.name} tidak lengkap: {kurang}")
    if "cbd_congestion_fee" not in df:
        df["cbd_congestion_fee"] = np.nan  # Tidak diketahui != nol.
        print(f"  [!] Kolom cbd_congestion_fee tidak tersedia di {path.name}; dibiarkan NaN")

    n = len(df)
    dup = int(df.duplicated().sum()) if CEK_DUPLIKAT else np.nan
    for c in ("tpep_pickup_datetime", "tpep_dropoff_datetime"):
        df[c] = pd.to_datetime(df[c], errors="coerce")
    for c in ("trip_distance", "fare_amount", "total_amount", "cbd_congestion_fee"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    awal = pd.Timestamp(TAHUN, bulan, 1)
    akhir = awal + pd.offsets.MonthBegin(1)
    waktu_valid = df["tpep_pickup_datetime"].ge(awal) & df["tpep_pickup_datetime"].lt(akhir)
    zona_valid = df["PULocationID"].isin(ZONA_VALID)
    demand_valid = waktu_valid & zona_valid
    durasi = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds()/60
    revenue_valid = (demand_valid & df["fare_amount"].ge(0) & df["total_amount"].ge(0)
                     & df["trip_distance"].ge(0) & durasi.gt(0))
    # Prototipe revenue: valid secara tanda dan durasi; batas outlier dianalisis, bukan langsung dibuang.
    missing = df.isna().sum()
    flag_n = pd.Series({ATURAN_BUANG[0]: int((~waktu_valid).sum()),
                        ATURAN_BUANG[1]: int((waktu_valid & ~zona_valid).sum()),
                        "Tidak valid untuk revenue (tidak dibuang dari demand)": int((demand_valid & ~revenue_valid).sum())})
    sampel_raw = df[["trip_distance", "fare_amount", "total_amount"]].sample(
        n=min(n, MAX_SAMPEL_BULAN), random_state=SEED+bulan).assign(kondisi="Sebelum filter")
    # Gunakan semua pickup yang memenuhi syarat demand.
    v = df.loc[demand_valid, ["tpep_pickup_datetime", "PULocationID", "DOLocationID",
                              "total_amount", "cbd_congestion_fee", "trip_distance", "fare_amount"]].copy()
    v["jam"] = v["tpep_pickup_datetime"].dt.floor("h")
    v["cbd_known"] = v["cbd_congestion_fee"].notna().astype("int8")
    v["cbd"] = (v["cbd_congestion_fee"] > 0).astype("int8")
    v["revenue_eligible"] = revenue_valid.loc[v.index].to_numpy()
    v["revenue_valid_amount"] = v["total_amount"].where(v["revenue_eligible"], 0).fillna(0)
    zh = (v.groupby(["PULocationID", "jam"], observed=True)
            .agg(trips=("PULocationID", "size"), revenue=("revenue_valid_amount", "sum"),
                 cbd_trips=("cbd", "sum"), cbd_known=("cbd_known", "sum"))
            .reset_index())
    od_ok = v["DOLocationID"].isin(ZONA_VALID)
    od = (v.loc[od_ok].groupby(["PULocationID", "DOLocationID"], observed=True)
            .agg(trips=("PULocationID", "size"), revenue=("revenue_valid_amount", "sum"),
                 cbd_trips=("cbd", "sum"), cbd_known=("cbd_known", "sum"))
            .reset_index())
    sampel_clean = v[["trip_distance", "fare_amount", "total_amount"]].sample(
        n=min(len(v), MAX_SAMPEL_BULAN), random_state=SEED+bulan).assign(kondisi="Sesudah filter demand")
    sampel_durasi = durasi.loc[revenue_valid].sample(
        n=min(int(revenue_valid.sum()), MAX_SAMPEL_BULAN), random_state=SEED+bulan).to_numpy()
    ringkas = dict(bulan=bulan, baris_mentah=n, baris_valid=int(demand_valid.sum()),
                   baris_revenue_valid=int(revenue_valid.sum()), duplikat=dup,
                   pct_cbd=(v["cbd"].sum()/v["cbd_known"].sum()*100 if v["cbd_known"].sum() else np.nan),
                   transaksi_cbd_teramati=int(v["cbd_known"].sum()))
    del df, v
    gc.collect()
    return dict(ringkas=ringkas, miss=missing, flag_n=flag_n, zh=zh, od=od,
                sampel=pd.concat([sampel_raw, sampel_clean], ignore_index=True),
                durasi=sampel_durasi)

hasil = {}
for b in BULAN:
    print(f"Memproses bulan {b:02d} ...", end=" ", flush=True)
    hasil[b] = proses_bulan(b)
    r = hasil[b]["ringkas"]
    print(f"{r['baris_mentah']:,} mentah -> {r['baris_valid']:,} demand valid")
ringkas = pd.DataFrame([h["ringkas"] for h in hasil.values()]).set_index("bulan")
miss_all = pd.concat([h["miss"] for h in hasil.values()], axis=1).sum(axis=1)
flag_df = pd.DataFrame({b: h["flag_n"] for b,h in hasil.items()}).T
zh_all = pd.concat([h["zh"] for h in hasil.values()], ignore_index=True)
zh_all["jam"] = pd.to_datetime(zh_all["jam"])
od_all = pd.concat([h["od"] for h in hasil.values()], ignore_index=True).groupby(
    ["PULocationID", "DOLocationID"], as_index=False)[["trips", "revenue", "cbd_trips", "cbd_known"]].sum()
sampel_dist = pd.concat([h["sampel"] for h in hasil.values()], ignore_index=True)
sampel_durasi = np.concatenate([h["durasi"] for h in hasil.values()])
N_MENTAH = int(ringkas["baris_mentah"].sum())
N_VALID  = int(ringkas["baris_valid"].sum())
assert int(zh_all["trips"].sum()) == N_VALID, "Total agregasi tidak sesuai demand valid!"
print(f"\nTotal: {N_MENTAH:,} mentah | {N_VALID:,} pickup valid ({N_VALID/N_MENTAH:.2%})")
print("Total hasil groupby cocok:", int(zh_all["trips"].sum()))
display(ringkas)
display(flag_df)
if CEK_DUPLIKAT: print("Jumlah baris identik (audit, belum dihapus):", int(ringkas["duplikat"].sum()))

# Tabel zona dipakai oleh EDA 2, 4, dan 5.
zone_tot = zh_all.groupby("PULocationID").agg(
    trips=("trips", "sum"), revenue=("revenue", "sum"),
    cbd_trips=("cbd_trips", "sum"), cbd_known=("cbd_known", "sum")
).reset_index().rename(columns={"PULocationID":"LocationID"})
zone_tot = zone_tot.merge(lookup[["LocationID", "Borough", "Zone", "label"]], on="LocationID", how="left", validate="many_to_one")
zone_tot["Borough"] = zone_tot["Borough"].fillna("Tidak diketahui")
zone_tot["share"] = zone_tot["trips"] / N_VALID * 100
zone_tot["rata_total"] = zone_tot["revenue"] / zone_tot["trips"]  # Bukan rata-rata transaksi revenue-valid.
zone_tot = zone_tot.sort_values("trips", ascending=False).reset_index(drop=True)
assert zone_tot["trips"].sum() == N_VALID

Memproses bulan 01 ... 3,724,889 mentah -> 3,718,952 demand valid
Memproses bulan 02 ... 3,399,866 mentah -> 3,394,829 demand valid
Memproses bulan 03 ... 3,952,451 mentah -> 3,946,183 demand valid
Memproses bulan 04 ... 3,831,240 mentah -> 3,825,196 demand valid
Memproses bulan 05 ... 

## EDA 1 - Pola Waktu: Profil Jam dan Heatmap Hari × Jam

**RQ1:** kapan pickup memuncak, bagaimana pola harian dan bulanan, dan apakah akhir pekan berbeda dari hari kerja?

> Timestamp sumber dalam **waktu lokal New York tanpa offset**. Jam `2026-03-08 02:00` saat DST mulai tidak boleh dihitung nol. Untuk Fase 4A, timestamp harus diberi model zona waktu yang memadai, termasuk penanganan jam yang berulang ketika DST berakhir (di luar periode Jan–Jun).

In [ ]:
awal_data = pd.Timestamp(TAHUN, BULAN[0], 1)
akhir_data = pd.Timestamp(TAHUN, BULAN[-1], 1) + pd.offsets.MonthBegin(1)
rentang = pd.date_range(awal_data, akhir_data, freq="h", inclusive="left")
ts = zh_all.groupby("jam")["trips"].sum().reindex(rentang, fill_value=0)
# DST start 2026: 8 Maret 02.00-02.59 tidak ada dalam waktu lokal NYC.
DST_HILANG = pd.Timestamp("2026-03-08 02:00")
if DST_HILANG in ts.index:
    if ts.loc[DST_HILANG] != 0:
        print("[!] Ada catatan pada jam lokal DST yang tidak tersedia:", ts.loc[DST_HILANG], "periksa timestamp sumber")
    ts = ts.drop(DST_HILANG)
tsd = ts.rename("trips").to_frame()
tsd["tanggal"] = tsd.index.normalize()
tsd["jam_hari"] = tsd.index.hour
tsd["hari"] = tsd.index.dayofweek
tsd["akhir_pekan"] = tsd["hari"] >= 5
tsd["libur"] = tsd["tanggal"].isin(tanggal_libur)
tsd["tipe_hari"] = np.where(tsd["libur"], "Libur nasional", np.where(tsd["akhir_pekan"], "Akhir pekan", "Hari kerja"))
N_JAM, N_HARI = len(tsd), tsd["tanggal"].nunique()
print("Jumlah jam valid:", N_JAM, "| Hari:", N_HARI)
harian = tsd.groupby("tanggal").agg(
    trips=("trips", "sum"), akhir_pekan=("akhir_pekan", "first"), libur=("libur", "first"))

prof = tsd.groupby(["tipe_hari", "jam_hari"])["trips"].agg(
    rata="mean", q1=lambda x:x.quantile(.25), q3=lambda x:x.quantile(.75)).reset_index()
warna = {"Hari kerja":"#4C72B0", "Akhir pekan":"#DD8452", "Libur nasional":"#55A868"}
fig, ax = plt.subplots(figsize=(12, 4.5))
for tipe, g in prof.groupby("tipe_hari"):
    ax.plot(g["jam_hari"], g["rata"], marker="o", ms=3.5, color=warna.get(tipe), label=tipe)
    ax.fill_between(g["jam_hari"], g["q1"], g["q3"], color=warna.get(tipe), alpha=.12)
ax.set_xticks(range(24)); ax.yaxis.set_major_formatter(mtick.FuncFormatter(ribuan))
ax.set(xlabel="Jam (waktu lokal NYC)", ylabel="Rata-rata pickup/jam", title="Profil pickup: hari kerja vs akhir pekan vs libur")
ax.legend(); plt.tight_layout(); simpan("eda1_profil_jam"); plt.show()

pv = tsd.pivot_table(index="hari", columns="jam_hari", values="trips", aggfunc="mean")
pv.index = [NAMA_HARI[i] for i in pv.index]
fig, ax = plt.subplots(figsize=(14, 4.2))
sns.heatmap(pv, cmap="YlOrRd", ax=ax, cbar_kws={"label":"Rata-rata pickup per jam"})
ax.set(xlabel="Jam", ylabel="", title="Heatmap rata-rata pickup: hari × jam")
plt.tight_layout(); simpan("eda1_heatmap_hari_jam"); plt.show()

rata_jam = tsd.groupby("jam_hari")["trips"].mean()
rata_hari = tsd.groupby("hari")["trips"].mean()
print(f"Jam puncak: {rata_jam.idxmax():02d}:00 ({rata_jam.max():,.0f} pickup/jam)")
print(f"Jam tersepi: {rata_jam.idxmin():02d}:00 ({rata_jam.min():,.0f} pickup/jam)")
print("Hari dengan rata-rata pickup tertinggi:", NAMA_HARI[int(rata_hari.idxmax())])

# Mann-Whitney atas total pickup HARIAN, tanpa hari libur; hasil merupakan asosiasi, bukan kausal.
d_nl = harian.loc[~harian["libur"]]
a,b = d_nl.loc[d_nl["akhir_pekan"], "trips"], d_nl.loc[~d_nl["akhir_pekan"], "trips"]
if len(a) and len(b):
    u,p = stats.mannwhitneyu(a, b, alternative="two-sided")
    cliff = 2*u/(len(a)*len(b)) - 1
    print(f"Mann-Whitney weekend vs weekday: U={u:,.0f}, p={p:.3g}, Cliff's delta={cliff:.3f}")
else: u=p=cliff=np.nan

# TAMBAHAN: grafik pickup per bulan dari semua bulan.
fig, ax = plt.subplots(figsize=(8,4))
ax.bar([NAMA_BULAN[i] for i in ringkas.index], ringkas["baris_valid"], color="#4C72B0")
ax.set(xlabel="Bulan 2026", ylabel="Jumlah pickup valid", title="Jumlah pickup per bulan (Jan–Jun 2026)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(ribuan))
plt.tight_layout(); simpan("eda1_pickup_bulanan"); plt.show()
print("Pickup valid per bulan:")
display(ringkas[["baris_mentah", "baris_valid", "baris_revenue_valid"]])

**Catatan interpretasi — RQ1:** laporkan jam puncak dan paling sepi, pola hari kerja/akhir pekan/libur, bulan dengan volume tertinggi, dan p-value serta arah perbedaan weekend. Bandingkan kalender eksternal Anggota 4 sebelum finalisasi kesimpulan.

## EDA 2 - Konsentrasi Permintaan Antar Zona (Top-K, Pareto, Gini)

**RQ4:** zona manakah yang memusatkan sebagian besar pickup? Analisis konsentrasi bersifat **deskriptif**, bukan pengujian hotspot spasial (itu Fase 4B).

In [ ]:
t15 = zone_tot.head(15)
fig, ax = plt.subplots(figsize=(9,6))
sns.barplot(data=t15, x="trips", y="label", hue="Borough", dodge=False, ax=ax)
ax.set(xlabel="Total pickup", ylabel="", title="Top 15 zona penjemputan")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(ribuan))
plt.tight_layout(); simpan("eda2_top15_zona"); plt.show()

# Hitung konsentrasi dari semua zona valid, termasuk zona tanpa pickup (nol).
semua_zona = sorted(ZONA_VALID)
zt_full = zone_tot.set_index("LocationID")["trips"].reindex(semua_zona, fill_value=0).sort_values(ascending=False)
kum = zt_full.cumsum()/zt_full.sum()*100
K_LIST = [k for k in [5,10,20,50,100] if k<=len(zt_full)]
cov = pd.DataFrame({"K":K_LIST})
cov["% zona"] = cov["K"]/len(zt_full)*100
cov["% pickup tertangkap"] = [kum.iloc[k-1] for k in K_LIST]
cov["lift vs rata-rata"] = cov["% pickup tertangkap"]/cov["% zona"]
display(cov.round(2))
xs = np.sort(zt_full.to_numpy(dtype=float))
n = len(xs)
GINI = (2*np.sum(np.arange(1,n+1)*xs)/(n*xs.sum())) - (n+1)/n
print(f"Gini antar zona = {GINI:.3f}")
fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.plot(np.arange(1,n+1), kum.values, color="#C44E52", lw=2)
ax.plot([0,n], [0,100], ls="--", color="grey", label="Distribusi merata")
for k in (10,20,50):
    if k<=n:
        ax.scatter([k], [kum.iloc[k-1]], c="black", zorder=5)
        ax.annotate(f"Top-{k}: {kum.iloc[k-1]:.0f}%", (k,kum.iloc[k-1]), xytext=(7,-12), textcoords="offset points")
ax.set(xlabel="Jumlah zona (dari yang teramai)", ylabel="% kumulatif pickup", title="Kurva Pareto pickup antar zona")
ax.legend(); plt.tight_layout(); simpan("eda2_pareto"); plt.show()

# Lengkapi kerangka zona-jam agar nol valid dihitung benar; jangan hitung jam DST hilang.
grid_count = len(ZONA_VALID) * N_JAM
slot_terisi = zh_all.loc[zh_all["jam"].isin(tsd.index), ["PULocationID", "jam"]].drop_duplicates()
PCT_NOL = (1-len(slot_terisi)/grid_count)*100
print(f"Estimasi zona-jam tanpa pickup: {PCT_NOL:.1f}% dari {grid_count:,} slot.")

# TAMBAHAN: distribusi jumlah pickup berdasarkan borough.
borough_tot = zone_tot.groupby("Borough")["trips"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9,4.5))
sns.barplot(x=borough_tot.values, y=borough_tot.index, color="#4C72B0", ax=ax)
ax.set(xlabel="Total pickup", ylabel="", title="Distribusi pickup menurut borough")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(ribuan))
plt.tight_layout(); simpan("eda2_pickup_borough"); plt.show()
display(pd.DataFrame({"pickup":borough_tot, "persen":borough_tot/N_VALID*100}).round(2))

**Catatan interpretasi — RQ4:** catat zona teramai, kontribusi Top-10/Top-20, nilai Gini, borough dominan, dan kesenjangan permintaan antar zona. Proporsi nol dari EDA adalah indikasi; grid final diverifikasi lagi pada Fase 4A.

## EDA 3 - Arus Asal-Tujuan Antar Borough

**RQ4:** ke mana perjalanan dari masing-masing borough mengarah? Distribusi ini menunjukkan perjalanan yang **tercatat**, bukan rute GPS aktual.

In [ ]:
od_b = od_all.assign(PU_b=od_all["PULocationID"].map(b_map).fillna("Tidak diketahui"),
                     DO_b=od_all["DOLocationID"].map(b_map).fillna("Tidak diketahui"))
mat = od_b.pivot_table(index="PU_b", columns="DO_b", values="trips", aggfunc="sum", fill_value=0)
mat_pct = mat.div(mat.sum(axis=1).replace(0,np.nan), axis=0)*100
fig, ax = plt.subplots(figsize=(9,5.2))
sns.heatmap(mat_pct, annot=True, fmt=".1f", cmap="Blues", ax=ax, cbar_kws={"label":"% dari pickup borough asal"})
ax.set(xlabel="Borough tujuan", ylabel="Borough asal", title="Matriks asal-tujuan antar borough (% per baris)")
plt.tight_layout(); simpan("eda3_od_borough"); plt.show()
intra = od_all.loc[od_all["PULocationID"]==od_all["DOLocationID"],"trips"].sum()/od_all["trips"].sum()*100
dalam_borough = np.trace(mat.reindex(index=mat.index,columns=mat.index,fill_value=0).values)/mat.values.sum()*100
print(f"Perjalanan dalam zona yang sama: {intra:.1f}%")
print(f"Perjalanan dalam borough yang sama: {dalam_borough:.1f}%")
top_od = od_all.nlargest(10,"trips").copy()
top_od["Rute"] = [f"{nama_zona(a)} -> {nama_zona(b)}" for a,b in zip(top_od["PULocationID"],top_od["DOLocationID"])]
top_od["% total OD valid"] = top_od["trips"]/od_all["trips"].sum()*100
display(top_od[["Rute","trips","% total OD valid"]].round(2))
print("Catatan: perjalanan tanpa zona dropoff valid tidak termasuk dalam matriks OD.")

### Tambahan EDA 3 — Peta pickup centroid (HTML) dan garis asal-tujuan

Peta menggunakan **centroid zona TLC**, bukan titik GPS asli perjalanan. File dibuat di `out/html/`. Coba tabel centroid bila tersedia, jika tidak hitung dari shapefile `datasets/zone/taxi_zones/taxi_zones.shp` (memerlukan `geopandas`). Hitung centroid dalam CRS proyeksi lalu konversi ke WGS84, bukan langsung pada koordinat lon/lat.

In [ ]:
try:
    import folium
    CENTROID_CSV = DATA_DIR / "zone" / "zone_centroid_lookup.csv"
    if CENTROID_CSV.exists():
        centroid = pd.read_csv(CENTROID_CSV)
        centroid = centroid.rename(columns={"longitude":"longitude","latitude":"latitude"})
        if "LocationID" not in centroid.columns or not {"longitude","latitude"}.issubset(centroid.columns):
            raise ValueError("Kolom centroid CSV harus LocationID, longitude, latitude")
    else:
        import geopandas as gpd
        SHAPEFILE = DATA_DIR / "zone" / "taxi_zones" / "taxi_zones.shp"
        if not SHAPEFILE.exists():
            raise FileNotFoundError(f"Shapefile tidak ditemukan: {SHAPEFILE}")
        zones_gdf = gpd.read_file(SHAPEFILE)
        zones_proj = zones_gdf.to_crs(epsg=2263)
        titik_wgs84 = gpd.GeoSeries(zones_proj.geometry.centroid, crs=zones_proj.crs).to_crs(epsg=4326)
        centroid = pd.DataFrame({"LocationID":pd.to_numeric(zones_gdf["LocationID"],errors="coerce"),
                                 "longitude":titik_wgs84.x.to_numpy(), "latitude":titik_wgs84.y.to_numpy()})
        centroid.dropna().to_csv(CENTROID_CSV, index=False)
        print("Tabel centroid dibuat:", CENTROID_CSV)
    centroid["LocationID"] = pd.to_numeric(centroid["LocationID"],errors="coerce")
    centroid = centroid.dropna(subset=["LocationID","longitude","latitude"]).drop_duplicates("LocationID")
    zmap = zone_tot.merge(centroid[["LocationID","longitude","latitude"]],on="LocationID",how="inner")
    peta = folium.Map(location=[40.73,-73.94],zoom_start=11,tiles="CartoDB positron")
    # Radius sengaja dibatasi agar zona ramai tetap terlihat dan map tidak berat.
    for _,r in zmap.iterrows():
        folium.CircleMarker(location=[r["latitude"],r["longitude"]],
            radius=float(np.clip(2+np.sqrt(r["trips"])/100,2,15)),
            fill=True, fill_opacity=.6, weight=.5,
            tooltip=f"{r['label']}: {int(r['trips']):,} pickup ({r['Borough']})").add_to(peta)
    lokasi_peta = HTML_DIR/"eda3_peta_pickup_centroid.html"
    peta.save(str(lokasi_peta)); print("Peta pickup disimpan:",lokasi_peta.resolve())

    # Pola OD: garis centroid pada 25 rute terbesar (bukan lintasan GPS jalan).
    pu_xy = centroid.rename(columns={"LocationID":"PULocationID","longitude":"pu_lon","latitude":"pu_lat"})
    do_xy = centroid.rename(columns={"LocationID":"DOLocationID","longitude":"do_lon","latitude":"do_lat"})
    od_coord = od_all.nlargest(25,"trips").merge(pu_xy,on="PULocationID",how="left").merge(do_xy,on="DOLocationID",how="left")
    od_coord = od_coord.dropna(subset=["pu_lon","pu_lat","do_lon","do_lat"])
    peta_od = folium.Map(location=[40.73,-73.94],zoom_start=11,tiles="CartoDB positron")
    for _,r in od_coord.iterrows():
        folium.PolyLine(locations=[[r["pu_lat"],r["pu_lon"]],[r["do_lat"],r["do_lon"]]],
            color="#C44E52",weight=float(np.clip(np.sqrt(r["trips"])/75,1,7)),opacity=.6,
            tooltip=f"{nama_zona(r['PULocationID'])} -> {nama_zona(r['DOLocationID'])} | {int(r['trips']):,} trip").add_to(peta_od)
    peta_od.save(str(HTML_DIR/"eda3_peta_od_centroid.html"))
    print("Peta OD disimpan:",(HTML_DIR/"eda3_peta_od_centroid.html").resolve())
except (ImportError, FileNotFoundError, KeyError, ValueError) as exc:
    print("[!] Peta HTML belum dibuat:",exc)
    print("Cek shapefile/centroid CSV dan install: pip install folium geopandas")

**Catatan interpretasi — RQ4:** jelaskan arus Manhattan–outer borough, rute OD teratas, dan keterbatasan data centroid. Garis OD adalah koneksi zona asal–tujuan, **bukan** rute kendaraan sebenarnya.

## EDA 4 - Jejak Congestion Pricing (`cbd_congestion_fee`)

**Deskriptif saja.** Persentase dihitung dari transaksi **yang nilai fee-nya teramati**, sehingga nilai tidak tersedia (NaN) tidak disalahartikan sebagai nol. Tidak ada klaim dampak kausal kebijakan.

In [ ]:
zh_all["jam_hari"] = zh_all["jam"].dt.hour
cbd_jam = zh_all.groupby("jam_hari")[["cbd_trips","cbd_known"]].sum()
cbd_jam["pct"] = cbd_jam["cbd_trips"]/cbd_jam["cbd_known"].replace(0,np.nan)*100
cbd_pu = zone_tot.groupby("Borough")[["cbd_trips","cbd_known"]].sum()
cbd_pu["pct"] = cbd_pu["cbd_trips"]/cbd_pu["cbd_known"].replace(0,np.nan)*100
cbd_do = od_b.groupby("DO_b")[["cbd_trips","cbd_known"]].sum()
cbd_do["pct"] = cbd_do["cbd_trips"]/cbd_do["cbd_known"].replace(0,np.nan)*100
fig,ax = plt.subplots(1,3,figsize=(17,4))
ax[0].bar([NAMA_BULAN[b] for b in ringkas.index],ringkas["pct_cbd"],color="#4C72B0")
ax[0].set(title="% berbiaya CBD per bulan",ylabel="% dari fee teramati")
ax[1].plot(cbd_jam.index,cbd_jam["pct"],marker="o",color="#C44E52")
ax[1].set(title="% berbiaya CBD menurut jam pickup",xlabel="Jam")
ax[1].set_xticks(range(0,24,3))
cbd_pu["pct"].sort_values().plot(kind="barh",ax=ax[2],color="#55A868")
ax[2].set(title="% berbiaya CBD menurut borough asal",xlabel="%",ylabel="")
plt.tight_layout(); simpan("eda4_cbd_fee"); plt.show()
print("Menurut borough tujuan:")
display(cbd_do[["pct"]].sort_values("pct",ascending=False).round(2))
print("Zona asal dengan % CBD tertinggi (min. 5.000 data fee teramati):")
cbd_zona = zone_tot[zone_tot["cbd_known"]>=5000].copy()
cbd_zona["pct_cbd"] = cbd_zona["cbd_trips"]/cbd_zona["cbd_known"]*100
display(cbd_zona.nlargest(10,"pct_cbd")[["label","Borough","trips","pct_cbd"]].round(2))

**Catatan interpretasi:** bandingkan proporsi CBD pada bulan, jam, dan borough. Karena semua data 2026 adalah sesudah penerapan kebijakan, jangan menyimpulkan efek sebelum–sesudah atau sebab-akibat.

## EDA 5 - Indeks Potensi Pendapatan per Zona

Mempertahankan perbandingan **peringkat pickup vs nilai transaksi**. Agar tidak salah mengartikan, nilai `revenue` di sini berasal dari transaksi revenue-valid; nilainya tidak boleh disebut total seluruh perjalanan jika banyak transaksi tidak valid.

In [ ]:
zone_tot["rank_pickup"] = zone_tot["trips"].rank(ascending=False)
zone_tot["rank_revenue"] = zone_tot["revenue"].rank(ascending=False)
rho,p_rho = stats.spearmanr(zone_tot["rank_pickup"],zone_tot["rank_revenue"])
print(f"Spearman ranking pickup vs revenue-valid: rho={rho:.3f}, p={p_rho:.2e}")
zone_tot["pickup_per_jam"] = zone_tot["trips"]/N_JAM
fig,ax = plt.subplots(1,2,figsize=(16,6))
r15 = zone_tot.nlargest(15,"revenue")
sns.barplot(x=r15["revenue"]/1e6,y=r15["label"],ax=ax[0],color="#C44E52")
ax[0].set(title="Top 15 zona menurut pendapatan valid",xlabel="Pendapatan valid (juta USD)",ylabel="")
aktif = zone_tot[(zone_tot["pickup_per_jam"]>0) & (zone_tot["revenue"]>0)]
ax[1].scatter(aktif["pickup_per_jam"],aktif["rata_total"],
    s=np.sqrt(aktif["revenue"])/8,alpha=.55,
    c=np.log10(aktif["revenue"].clip(lower=1)),cmap="viridis")
ax[1].set_xscale("log")
ax[1].set(xlabel="Rata-rata pickup/jam (log)",ylabel="Revenue-valid per pickup (USD)",
          title="Volume vs nilai transaksi valid")
for _,r in aktif.nlargest(6,"revenue").iterrows():
    ax[1].annotate(r["Zone"],(r["pickup_per_jam"],r["rata_total"]),fontsize=8,xytext=(4,4),textcoords="offset points")
plt.tight_layout(); simpan("eda5_indeks_pendapatan"); plt.show()
zone_tot["selisih_rank"] = zone_tot["rank_pickup"]-zone_tot["rank_revenue"]
print("Zona yang rank revenue validnya lebih baik daripada rank pickup:")
display(zone_tot[zone_tot["trips"]>=1000].nlargest(8,"selisih_rank")[["label","Borough","rank_pickup","rank_revenue","rata_total"]].round(2))

**Catatan interpretasi:** jelaskan apakah peringkat volume dan nilai transaksi sejalan. **Hati-hati:** yang tersedia adalah nilai transaksi tercatat, bukan keuntungan operasional bersih.

## EDA Tambahan Wajib — Distribusi Perjalanan dan Evaluasi Cleaning

Butir TODO yang belum ada di notebook lama: jarak, durasi, tarif; perbandingan **sebelum vs sesudah** filter. Demi efisiensi memori, histogram memakai **sampel reproducible per bulan**. Total agregasi tetap dihitung dari seluruh baris. Data yang ekstrem hanya dibatasi **untuk tampilan grafik** (bukan dihapus permanen), dan persentil dicatat.

In [ ]:
# Histogram statistik perjalanan: sampel 20.000 per bulan, bukan angka total populasi.
kolom_plot = {"trip_distance":"Jarak perjalanan (mil)",
              "fare_amount":"Tarif dasar (USD)","total_amount":"Total biaya (USD)"}
fig,axes = plt.subplots(1,3,figsize=(17,4.5))
for ax,(col,label) in zip(axes,kolom_plot.items()):
    s = pd.to_numeric(sampel_dist.loc[sampel_dist["kondisi"]=="Sesudah filter demand",col],errors="coerce")
    hi = s.quantile(.99)
    ss = s[s.between(0,hi)]
    sns.histplot(ss,bins=50,ax=ax,color="#4C72B0")
    ax.set(title=f"Distribusi {label} (hingga P99)",xlabel=label,ylabel="Frekuensi sampel")
    print(f"{col}: median={s.median():.2f}, P99={hi:.2f}, missing={s.isna().mean():.2%}, negatif={(s<0).mean():.2%}")
plt.tight_layout(); simpan("eda_tambahan_distribusi_perjalanan"); plt.show()
fig,ax = plt.subplots(figsize=(7,4))
d = pd.Series(sampel_durasi).replace([np.inf,-np.inf],np.nan).dropna()
lim = d.quantile(.99)
sns.histplot(d[d.between(0,lim)],bins=45,ax=ax,color="#DD8452")
ax.set(title="Durasi perjalanan revenue-valid (hingga P99)",xlabel="Durasi (menit)",ylabel="Frekuensi sampel")
plt.tight_layout(); simpan("eda_tambahan_distribusi_durasi"); plt.show()

# Before-after cleaning: hitung distribusi, bukan hanya jumlah baris.
fig,axs = plt.subplots(1,3,figsize=(17,4.5))
for ax,(col,label) in zip(axs,kolom_plot.items()):
    raw = sampel_dist.loc[sampel_dist["kondisi"]=="Sebelum filter",col].dropna()
    bersih = sampel_dist.loc[sampel_dist["kondisi"]=="Sesudah filter demand",col].dropna()
    lim = pd.concat([raw,bersih]).quantile(.99)
    ax.hist(raw[raw.between(0,lim)], bins=40, density=True, histtype="step", color="#4C72B0", label="Sebelum")
    ax.hist(bersih[bersih.between(0,lim)], bins=40, density=True, histtype="step", color="#DD8452", label="Sesudah demand")
    ax.set(title=f"Sebelum vs sesudah: {label}",xlabel=label,ylabel="Kepadatan sampel");ax.legend()
plt.tight_layout();simpan("eda_tambahan_sebelum_sesudah_cleaning");plt.show()
print("Jumlah baris setiap bulan sebelum/sesudah filter demand:")
display(ringkas[["baris_mentah","baris_valid","baris_revenue_valid"]])
print("[!] Jangan menyimpulkan dampak aturan cleaning final dari filter prototipe ini.")

## EDA Tambahan Wajib — Distribusi Target untuk Regresi

Target `pickup_count` zona–jam merupakan dasar prediksi **t+1**. Pada fase EDA, kita memeriksa bentuk distribusi pickup zona–jam, termasuk nol, skewness, outlier, dan kemungkinan `log1p`. Fase 4A tetap bertanggung jawab menghasilkan `demand_zone_hour` final.

In [ ]:
# Mengambil semua zona x jam valid tanpa membuat tabel join penuh yang besar.
# Periode Jan–Jun 2026: tidak ada DST fall-back; DST spring-forward dikeluarkan.
n_slot = len(ZONA_VALID)*len(tsd)
observed = (zh_all.loc[zh_all["jam"].isin(tsd.index)]
            .groupby(["PULocationID","jam"])["trips"].sum())
nonzero = observed.to_numpy(dtype=np.float64)
zero_count = n_slot-len(nonzero)
if zero_count<0: raise AssertionError("Jumlah pasangan zona-jam melampaui kapasitas grid")
y = np.concatenate([nonzero,np.zeros(zero_count,dtype=float)])
y_skew = float(stats.skew(y)) if y.std()>0 else 0.0
print(f"Jumlah slot zona-jam valid: {len(y):,}")
print(f"Proporsi nol: {np.mean(y==0):.2%} | Skewness: {y_skew:.3f}")
print(f"Mean: {np.mean(y):.2f} | Median: {np.median(y):.2f} | P99: {np.quantile(y,.99):.1f} | Max: {np.max(y):.0f}")
q1,q3 = np.quantile(y,[.25,.75]); iqr=q3-q1
batas_outlier = q3+1.5*iqr
print(f"Outlier IQR (> {batas_outlier:.1f} pickup): {np.mean(y>batas_outlier):.2%}")
fig,axs=plt.subplots(1,2,figsize=(12,4.2))
sns.histplot(y[y<=np.quantile(y,.99)],bins=50,ax=axs[0],color="#4C72B0")
axs[0].set(xlabel="Pickup per zona-jam",ylabel="Jumlah sel",title="Distribusi target sampai P99")
sns.histplot(np.log1p(y),bins=50,ax=axs[1],color="#55A868")
axs[1].set(xlabel="log1p(pickup_count)",ylabel="Jumlah sel",title="Distribusi target setelah log1p")
plt.tight_layout(); simpan("eda_target_pickup_count");plt.show()
print("Keputusan log1p belum final: perlu dikaji kembali saat Fase 6B dan metrik dihitung pada skala pickup asli.")

## Ringkasan Temuan dan Ekspor

Semua kesimpulan berikut **sementara sampai hasil dikonfirmasi pada `main_trips` final**. Selain output berbentuk teks, CSV dibuat untuk memudahkan cross-check dengan TODO/EDA UTS. **Uji statistik yang sudah ada perlu dibandingkan dengan hasil EDA UTS**, jika notebook/data UTS tersedia.

In [ ]:
print("RINGKASAN TEMUAN EDA (PROTOTIPE)")
print("-"*70)
print(f"Data: {N_MENTAH:,} baris mentah -> {N_VALID:,} demand valid ({N_VALID/N_MENTAH:.2%}).")
print(f"Jam puncak {rata_jam.idxmax():02d}:00, jam sepi {rata_jam.idxmin():02d}:00; hari rata-rata tertinggi: {NAMA_HARI[int(rata_hari.idxmax())]}.")
print(f"Top zona {zone_tot.iloc[0]['label']} ({zone_tot.iloc[0]['share']:.1f}% pickup); Top-{min(10,len(kum))} {kum.iloc[min(10,len(kum))-1]:.1f}%; Gini {GINI:.3f}.")
print(f"Pickup zona-jam bernilai nol: {np.mean(y==0):.1%}, skewness {y_skew:.2f}.")
print(f"OD dalam zona sama {intra:.1f}%; dalam borough sama {dalam_borough:.1f}%.")
print(f"Rata-rata persentase CBD antarbulan (unweighted): {ringkas['pct_cbd'].mean():.1f}% dari fee teramati.")
print(f"Spearman rank pickup vs revenue-valid: {rho:.2f}.")

# Export: angka ini berasal dari prototipe raw; jangan digunakan menggantikan hasil agregasi Fase 4A.
ringkas.to_csv(OUT_DIR/"eda_ringkasan_bulanan.csv")
zone_tot.to_csv(OUT_DIR/"eda_ringkasan_zona.csv",index=False)
cov.to_csv(OUT_DIR/"eda_coverage_topk.csv",index=False)
borough_tot.rename("pickup").to_csv(OUT_DIR/"eda_pickup_borough.csv")
od_all.to_csv(OUT_DIR/"eda_agregat_od_prototipe.csv",index=False)
flag_df.to_csv(OUT_DIR/"eda_audit_filter_prototipe.csv")
pd.DataFrame({"metrik":["total_mentah","total_demand_valid","persen_nol_target","skewness_target","gini_zona","mannwhitney_p"],
    "nilai":[N_MENTAH,N_VALID,float(np.mean(y==0)),y_skew,GINI,float(p)]}).to_csv(OUT_DIR/"eda_statistik_utama.csv",index=False)
print("Output tabel dan",len(list(FIG_DIR.glob("*.png"))),"grafik tersimpan di",OUT_DIR.resolve())
print("HTML tersimpan di",HTML_DIR.resolve())
print("GATE FASE 3: interpretasi harus ditulis dan statistik diverifikasi terhadap main_trips final + EDA UTS.")